In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score
import cv2
import os
from PIL import Image

os.chdir("/home/oriado/Files/meter_project")

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [3]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model = DigitCNN().to(device)

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_losses = []

for epoch in range(5):
    model.train()
    total_loss = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5  Loss: {avg_loss:.4f}")

Epoch 1/5  Loss: 0.1333
Epoch 2/5  Loss: 0.0437
Epoch 3/5  Loss: 0.0289
Epoch 4/5  Loss: 0.0216
Epoch 5/5  Loss: 0.0157


In [5]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 99.03%


In [6]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel("Передбачення")
plt.ylabel("Правильна відповідь")
plt.xticks(range(10))
plt.yticks(range(10))
plt.title("Confusion Matrix")

for i in range(10):
    for j in range(10):
        if cm[i][j] > cm.max() / 2:
            color = 'white'
        else:
            color = 'black'
        plt.text(j, i, str(cm[i][j]), ha='center', va='center', color=color)

plt.tight_layout()
plt.savefig("confusion_matrix_ocr.jpg")
plt.close()

In [7]:
plt.figure()
plt.plot(range(1, 6), train_losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.savefig("ocr_training_loss.jpg")
plt.close()

In [8]:
torch.save(model.state_dict(), "ocr_model.pt")

In [9]:
digit_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def read_number(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    scale = max(1, 80 // gray.shape[0])
    gray = cv2.resize(gray, (gray.shape[1] * scale, gray.shape[0] * scale))

    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h, w = thresh.shape
    digit_boxes = []

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        if cw > 5 and ch > h * 0.3 and cw < w * 0.5:
            digit_boxes.append((x, y, cw, ch))

    n = len(digit_boxes)
    for i in range(n):
        for j in range(0, n - i - 1):
            if digit_boxes[j][0] > digit_boxes[j + 1][0]:
                temp = digit_boxes[j]
                digit_boxes[j] = digit_boxes[j + 1]
                digit_boxes[j + 1] = temp

    if not digit_boxes:
        return "не розпізнано"

    result = ""
    model.eval()

    with torch.no_grad():
        for (x, y, cw, ch) in digit_boxes:
            digit = gray[y:y+ch, x:x+cw]
            digit = cv2.resize(digit, (28, 28))
            digit_pil = Image.fromarray(digit)
            tensor = digit_transform(digit_pil).unsqueeze(0).to(device)
            pred = model(tensor).argmax(1).item()
            result += str(pred)

    return result

In [10]:
crop = cv2.imread("example_crop.jpg")
if crop is not None:
    print(read_number(crop))

5555555
